# Build Sign Count Dataset

This notebook converts the merged long TCAV CSV into a sample-level wide dataset using `sign_count` features only.


In [ ]:
from pathlib import Path

import pandas as pd


In [ ]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
TCAV_ROOT = PROJECT_ROOT / 'redimnet' / 'tcav' / 'captum_tcav' / 'asvspoof5'
OUTPUT_SUBDIR = 'subset_20spk_20utts_per_spk_one_logistic_head'
OUTPUT_DIR = TCAV_ROOT / 'output' / OUTPUT_SUBDIR

INPUT_CSV = OUTPUT_DIR / 'merged_tcav_long.csv'
OUTPUT_CSV = OUTPUT_DIR / 'merged_tcav_sign_count_wide.csv'

# Set True if you want to exclude A12 from the concept-level dataset.
EXCLUDE_A12 = False

print('INPUT_CSV =', INPUT_CSV)
print('Exists =', INPUT_CSV.exists())


In [ ]:
df = pd.read_csv(INPUT_CSV)
print('Loaded rows =', len(df))
display(df.head())


In [ ]:
required_cols = {
    'utt_id',
    'speaker_id',
    'split',
    'source_partition',
    'system_id',
    'target_class',
    'concept_name',
    'sign_count',
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing columns: {sorted(missing)}')

if EXCLUDE_A12:
    df = df[df['system_id'] != 'A12'].copy()

df['binary_label'] = (df['target_class'] == 'spoof').astype(int)

print('Rows after filtering =', len(df))
print('Unique utterances =', df['utt_id'].nunique())
print('Systems =', sorted(df['system_id'].unique().tolist()))
print('Concepts =', sorted(df['concept_name'].unique().tolist()))


In [ ]:
meta_cols = ['utt_id', 'speaker_id', 'split', 'source_partition', 'system_id', 'target_class', 'binary_label']

wide_df = (
    df.pivot_table(
        index=meta_cols,
        columns='concept_name',
        values='sign_count',
        aggfunc='first',
    )
    .reset_index()
)

wide_df.columns.name = None
concept_cols = [c for c in wide_df.columns if c not in meta_cols]
wide_df = wide_df.sort_values(['source_partition', 'system_id', 'target_class', 'speaker_id', 'utt_id']).reset_index(drop=True)

print('Wide rows =', len(wide_df))
print('Feature columns =', len(concept_cols))
print('Missing feature values =', int(wide_df[concept_cols].isna().sum().sum()))
display(wide_df.head())


In [ ]:
summary_df = pd.DataFrame({
    'rows': [len(wide_df)],
    'utterances': [wide_df['utt_id'].nunique()],
    'speakers': [wide_df['speaker_id'].nunique()],
    'systems': [wide_df['system_id'].nunique()],
    'sign_count_features': [len(concept_cols)],
    'spoof_rows': [int((wide_df['binary_label'] == 1).sum())],
    'bonafide_rows': [int((wide_df['binary_label'] == 0).sum())],
})
display(summary_df)

display(
    wide_df.groupby(['split', 'target_class'])['utt_id']
    .nunique()
    .rename('n_utterances')
    .reset_index()
    .sort_values(['split', 'target_class'])
)


In [ ]:
wide_df.to_csv(OUTPUT_CSV, index=False)
print('Saved sign-count wide dataset to:', OUTPUT_CSV)
